In [39]:

import warnings
import torch
import numpy as np

from rich import print
from datasets import  final_extinctionrisk_dataframe, final_extinctionrisk_noth_dataframe
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold
from pytorch_tabular import TabularModel
from pytorch_tabular.models import TabNetModelConfig, TabTransformerConfig
from pytorch_tabular.config import (
    DataConfig,
    OptimizerConfig,
    TrainerConfig,
)

In [40]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device == "cuda":
    torch.set_float32_matmul_precision('high')

# With Human Threats, With Cross Validation

In [ ]:
model, data = final_extinctionrisk_dataframe()
categorical_data = data.drop(model.numeric, axis=1)
cat_col_names = list(categorical_data.columns)
num_col_names = model.numeric

mapping = {'Lower_risk':1, 'Higher_risk':2}
data['Extinction_risk'] = data['Extinction_risk'].map(mapping)

print(f"Data Shape: {data.shape} | # of cat cols: {len(cat_col_names)} | # of num cols: {len(num_col_names)}")
print(f"[bold dodger_blue2] Features: {num_col_names + cat_col_names}[/bold dodger_blue2]")
print(f"[bold purple4]Target: {model.label}[/bold purple4]")

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

Data Shape: (9053, 32) | # of cat cols: 14 | # of num cols: 18

 Features: ['Beak_length_culmen', 'Beak_depth', 'Tarsus_length', 'Wing_length', 'Hand_wing_index', 'Tail_length', 
'Minimum_latitude', 'Maximum_latitude', 'Minimum_elevation', 'Elevational_range', 'Maximum_elevation', 
'Habitat_breadth', 'Diet_breadth', 'Adult_survival_annual', 'Generation_length', 'Range_size', 'Body_mass', 
'Clutch_size', 'Order', 'Family', 'Agriculture', 'Hunting', 'Invasive_species', 'Climate_change', 
'Primary_lifestyle', 'Island_restricted_breeding', 'Latitudinal_range', 'Realm', 'Diet', 'Habitat', 'Migration', 
'Extinction_risk']

Target: Extinction_risk

In [42]:
data_config = DataConfig(
    target=[
        model.label
    ],  
    continuous_cols=num_col_names,
    categorical_cols=cat_col_names,
    num_workers = 0, 
    pin_memory=True
)
trainer_config = TrainerConfig(
    batch_size=1024,
    max_epochs=50,
    early_stopping_patience=10,
)
optimizer_config = OptimizerConfig()

model_config = TabNetModelConfig(
    n_d = 8,
    n_a = 8,
    n_steps = 3,
    gamma = 1.3,
    n_independent = 2,
    n_shared = 2,
    virtual_batch_size=128,
    mask_type = 'sparsemax',
    task="classification",
    head = 'LinearHead',
    embedding_dropout = 0.0,
    batch_norm_continuous_input = True,
    learning_rate = 0.001,
    seed = 42,
    metrics = ["accuracy"]
)

In [43]:
tabular_model = TabularModel(
    data_config=data_config,
    model_config=model_config,
    optimizer_config=optimizer_config,
    trainer_config=trainer_config,
    verbose=True
)

2026-03-17 12:12:22,708 - {pytorch_tabular.tabular_model:145} - INFO - Experiment Tracking is turned off


In [ ]:
#accuracy_score, f1_score, precision_score, recall_score
acc_metrics = []
f1_metrics = [] 
precision_metrics = []
recall_metrics = []

def _metrics(y_true, y_pred):
    y_t = y_true['Extinction_risk'].map(mapping)
    f1 = f1_score(y_t, y_pred["Extinction_risk_prediction"].values, average=None)
    precision = precision_score(y_t, y_pred["Extinction_risk_prediction"].values, average=None)
    recall = recall_score(y_t, y_pred["Extinction_risk_prediction"].values, average=None)
    acc = accuracy_score(y_t, y_pred["Extinction_risk_prediction"].values)

    acc_metrics.append(acc)
    f1_metrics.append(f1)
    precision_metrics.append(precision)
    recall_metrics.append(recall)

    return acc

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    cv_scores, oof_predictions = tabular_model.cross_validate(
        cv=10, train=data, metric=_metrics, return_oof=True, reset_datamodule=True
    )

2026-03-17 12:12:22,750 - {pytorch_tabular.tabular_model:2216} - INFO - Running Fold 1/10
2026-03-17 12:12:22,751 - {pytorch_tabular.tabular_model:547} - INFO - Preparing the DataLoaders
2026-03-17 12:12:22,761 - {pytorch_tabular.tabular_datamodule:527} - INFO - Setting up the datamodule for classification task
2026-03-17 12:12:22,816 - {pytorch_tabular.tabular_model:598} - INFO - Preparing the Model: TabNetModel
2026-03-17 12:12:22,876 - {pytorch_tabular.tabular_model:341} - INFO - Preparing the Trainer
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
2026-03-17 12:12:22,885 - {pytorch_tabular.tabular_model:677} - INFO - Training Started
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name             ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ _embedding_layer │ Identity         │      0 │ train │     0 │
│ 1 │ _backbone        │ TabNetBackbone   │ 24.9 K │ train │     0 │
│ 2 │ _head            │ Identity         │      0 │ train │     0 │
│ 3 │ loss             │ CrossEntropyLoss │      0 │ train │     0 │
└───┴──────────────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 24.9 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 24.9 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 122                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

`Trainer.fit` stopped: `max_epochs=50` reached.


...

┏━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name             ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ _embedding_layer │ Identity         │      0 │ train │     0 │
│ 1 │ _backbone        │ TabNetBackbone   │ 26.4 K │ train │     0 │
│ 2 │ _head            │ Identity         │      0 │ train │     0 │
│ 3 │ loss             │ CrossEntropyLoss │      0 │ train │     0 │
└───┴──────────────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 26.4 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 26.4 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 122                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

`Trainer.fit` stopped: `max_epochs=50` reached.


2026-03-17 12:18:42,051 - {pytorch_tabular.tabular_model:690} - INFO - Training the model completed
2026-03-17 12:18:42,051 - {pytorch_tabular.tabular_model:1531} - INFO - Loading the best model
2026-03-17 12:18:42,150 - {pytorch_tabular.tabular_model:2248} - INFO - Fold 10/10 score: 0.9513812154696133


In [45]:
print(f"KFold Mean: {np.mean(cv_scores)} | KFold SD: {np.std(cv_scores)}")

KFold Mean: 0.8592307879940971 | KFold SD: 0.2509712408368808

In [46]:
print(acc_metrics)

[
    0.11699779249448124,
    0.9536423841059603,
    0.890728476821192,
    0.988950276243094,
    0.9955801104972376,
    0.9281767955801105,
    0.9966850828729282,
    0.8939226519337017,
    0.876243093922652,
    0.9513812154696133
]